In [1]:
import pandas as pd

file_path = r"C:\Users\boopa\Desktop\DS\Drone\drone_delivery.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.info())
print(df.isnull().sum())

order_id                  0
drone_id               9137
drone_type             4989
drone_speed_kmph       1825
payload_weight_kg      8958
distance_km            9036
battery_efficiency     1479
climate_condition      5925
wind_speed_kmph           0
temperature_c             0
humidity_percent          0
source_area            5946
destination_area       5938
traffic_condition      4978
ETA_actual_min        11023
dtype: int64


In [2]:
# Separate numeric & categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numeric missing values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical missing values with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [3]:
df = df.drop(columns=['order_id', 'drone_id'])

In [4]:
df = pd.get_dummies(df, drop_first=True)

In [5]:
X = df.drop('ETA_actual_min', axis=1)
y = df['ETA_actual_min']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression MAE:", mean_absolute_error(y_test, y_pred_lr))
print("Linear Regression R2:", r2_score(y_test, y_pred_lr))

Linear Regression MAE: 5.9220957441026
Linear Regression R2: 0.011987806005108959


Random Forest

In [8]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest MAE:", mean_absolute_error(y_test, y_pred_rf))
print("Random Forest R2:", r2_score(y_test, y_pred_rf))

Random Forest MAE: 6.0931849
Random Forest R2: 0.038897605016023906


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # Output layer for regression
])

C:\Users\boopa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(
    optimizer='adam',
    loss='mae',   # Mean Absolute Error
    metrics=['mae']
)

In [12]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

10000/10000 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - loss: 4.0472 - mae: 4.0472 - val_loss: 4.1609 - val_mae: 4.1609


In [13]:
y_pred_dl = model.predict(X_test_scaled)

from sklearn.metrics import mean_absolute_error, r2_score

print("Deep Learning MAE:", mean_absolute_error(y_test, y_pred_dl))
print("Deep Learning R2:", r2_score(y_test, y_pred_dl))

Deep Learning MAE: 4.384190130330801
Deep Learning R2: -0.010256365340900686


In [15]:
import joblib

joblib.dump(rf, 'random_forest_model.pkl')
print("Model saved!")

joblib.dump(rf, r'C:\Users\boopa\Desktop\DS\Drone\random_forest_model.pkl')
joblib.dump(X_train.columns.tolist(), r'C:\Users\boopa\Desktop\DS\Drone\model_columns.pkl')

['C:\\Users\\boopa\\Desktop\\DS\\Drone\\model_columns.pkl']

copying all images in sigle folder

In [1]:
import os
import shutil

source_folders = [
    "fixed_wing_drone",
    "hybrid_drone",
    "multi_rotor_drones",
    "single_rotor_drone",
    "propeller_crack_drone",
    "wing_damage_drone",
    "missing_parts_drone"
]

destination = "dataset_all_images"

counter = 1

for folder in source_folders:
    for file in os.listdir(folder):
        if file.endswith(".png") or file.endswith(".jpg"):

            new_name = f"drone_{counter}.png"

            shutil.copy(
                os.path.join(folder, file),
                os.path.join(destination, new_name)
            )

            counter += 1

print("All images copied successfully")

All images copied successfully


from roboflow

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="P3jRAyMDd2us9grSk0Dj")
project = rf.workspace("boopathys-workspace").project("drone-detection-ab55w")
version = project.version(1)
dataset = version.download("yolov8")

Train the YOLO Model

In [2]:
from ultralytics import YOLO

# load model
model = YOLO("yolov8n.pt")

# train
model.train(
    data="drone-detection-1/data.yaml",
    epochs=50,
    imgsz=640
)

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000204A7C340C0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [2]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

results = model("drone-detection-1/test/images", show=True)

Results saved to C:\Users\boopa\Desktop\DS\Drone\runs\detect\predict


Connect Python to AWS RDS

In [7]:
!pip install cryptography


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\boopa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    port=3306
)

print("Connected successfully!")

RuntimeError: 'cryptography' package is required for sha256_password or caching_sha2_password auth methods

In [12]:
import sys
print(sys.executable)

C:\Users\boopa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


In [13]:
import sys
!{sys.executable} -m pip install cryptography


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\boopa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    port=3306
)

print("Connected successfully!")

Connected successfully!


Create Database and Table in RDS

In [2]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    port=3306
)

cursor = connection.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS drone_db")
cursor.execute("USE drone_db")

cursor.execute("""
CREATE TABLE IF NOT EXISTS detections (
    id INT AUTO_INCREMENT PRIMARY KEY,
    drone_type VARCHAR(50),
    confidence FLOAT,
    image_url TEXT,
    detected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

connection.commit()
print("Database and table created!")

Database and table created!


In [3]:
!pip install boto3


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\boopa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import boto3

s3 = boto3.client("s3")

bucket_name = "drone-detection-storage-boopathy"

file_path = "drone-detection-1/test/images/drone_128_png.rf.093df5ca0a582eb697a50efaeccab62d.jpg"

s3_key = "detections/drone_test.jpg"

# upload image
s3.upload_file(file_path, bucket_name, s3_key)

print("Upload successful!")

# generate image URL
image_url = f"https://{bucket_name}.s3.amazonaws.com/{s3_key}"
print("Image URL:", image_url)

Upload successful!
Image URL: https://drone-detection-storage-boopathy.s3.amazonaws.com/detections/drone_test.jpg


store metadata in RDS

In [10]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    database="drone_db",
    port=3306
)

cursor = connection.cursor()

print("Reconnected successfully!")

Reconnected successfully!


In [11]:
cursor.execute("""
INSERT INTO detections (drone_type, confidence, image_url)
VALUES (%s, %s, %s)
""", ("fixed_wing_drone", 0.92, image_url))

connection.commit()

print("Metadata stored in RDS!")

Metadata stored in RDS!


In [12]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    database="drone_db",
    port=3306
)

cursor = connection.cursor()

cursor.execute("SELECT * FROM detections")

rows = cursor.fetchall()

print("Rows found:", len(rows))

for row in rows:
    print(row)

Rows found: 1
(1, 'fixed_wing_drone', 0.92, 'https://drone-detection-storage-boopathy.s3.amazonaws.com/detections/drone_test.jpg', datetime.datetime(2026, 3, 10, 16, 40, 42))


In [13]:
cursor.execute("SELECT * FROM detections")
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'fixed_wing_drone', 0.92, 'https://drone-detection-storage-boopathy.s3.amazonaws.com/detections/drone_test.jpg', datetime.datetime(2026, 3, 10, 16, 40, 42))


Full Pipeline Script

In [1]:
from ultralytics import YOLO
import boto3
import pymysql
import glob
import os

# ----------------------------
# 1. CONFIGURATION
# ----------------------------

MODEL_PATH = "runs/detect/train/weights/best.pt"

IMAGE_PATH = "drone-detection-1/test/images/drone_128_png.rf.093df5ca0a582eb697a50efaeccab62d.jpg"

S3_BUCKET = "drone-detection-storage-boopathy"
S3_KEY = "detections/drone_result.jpg"

RDS_HOST = "drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com"
RDS_USER = "admin"
RDS_PASSWORD = "Drone1234"
RDS_DB = "drone_db"


# ----------------------------
# 2. LOAD YOLO MODEL
# ----------------------------

model = YOLO(MODEL_PATH)


# ----------------------------
# 3. RUN DETECTION
# ----------------------------

results = model(IMAGE_PATH, save=True)

detected_class = None
confidence = None

for r in results:
    if len(r.boxes) > 0:
        detected_class = model.names[int(r.boxes.cls[0])]
        confidence = float(r.boxes.conf[0])

print("Detected:", detected_class, confidence)


# ----------------------------
# 4. FIND SAVED DETECTION IMAGE
# ----------------------------

predict_folders = glob.glob("runs/detect/predict*")

latest_folder = max(predict_folders, key=os.path.getctime)

images = glob.glob(f"{latest_folder}/*.jpg")

output_image = images[0]

print("Detection image path:", output_image)


# ----------------------------
# 5. UPLOAD IMAGE TO S3
# ----------------------------

s3 = boto3.client("s3")

s3.upload_file(output_image, S3_BUCKET, S3_KEY)

image_url = f"https://{S3_BUCKET}.s3.amazonaws.com/{S3_KEY}"

print("Uploaded to S3:", image_url)


# ----------------------------
# 6. STORE METADATA IN RDS
# ----------------------------

connection = pymysql.connect(
    host=RDS_HOST,
    user=RDS_USER,
    password=RDS_PASSWORD,
    database=RDS_DB,
    port=3306
)

cursor = connection.cursor()

cursor.execute("""
INSERT INTO detections (drone_type, confidence, image_url)
VALUES (%s, %s, %s)
""", (detected_class, confidence, image_url))

connection.commit()

print("Metadata stored in RDS")

connection.close()

Metadata stored in RDS


In [1]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    database="drone_db",
    port=3306
)

cursor = connection.cursor()

print("Connected to RDS again")

Connected to RDS again


In [2]:
cursor.execute("SELECT * FROM detections")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'fixed_wing_drone', 0.92, 'https://drone-detection-storage-boopathy.s3.amazonaws.com/detections/drone_test.jpg', datetime.datetime(2026, 3, 10, 16, 40, 42))
(2, 'fixed_wing_drone', 0.457571, 'https://drone-detection-storage-boopathy.s3.amazonaws.com/detections/drone_result.jpg', datetime.datetime(2026, 3, 10, 16, 52, 5))


Fix table


In [4]:
import pymysql

try:

    connection = pymysql.connect(
        host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
        user="admin",
        password="Drone1234",
        database="drone_db",
        port=3306,
        connect_timeout=10
    )

    cursor = connection.cursor()

    cursor.execute("""
    ALTER TABLE detections
    ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    """)

    connection.commit()

    print("SUCCESS: created_at column added!")

except Exception as e:
    print("Error:", e)

finally:
    try:
        connection.close()
    except:
        pass

Error: (1060, "Duplicate column name 'created_at'")


In [3]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    database="drone_db",
    port=3306
)

cursor = connection.cursor()
cursor.execute("SELECT 1")

print("Connection successful!")

Connection successful!


In [5]:
import pymysql

connection = pymysql.connect(
    host="drone-detection-db.czyge4aiywpv.ap-south-1.rds.amazonaws.com",
    user="admin",
    password="Drone1234",
    database="drone_db",
    port=3306
)

cursor = connection.cursor()

cursor.execute("DESCRIBE detections")

for row in cursor.fetchall():
    print(row)

('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('drone_type', 'varchar(50)', 'YES', '', None, '')
('confidence', 'float', 'YES', '', None, '')
('image_url', 'text', 'YES', '', None, '')
('detected_at', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')
('created_at', 'timestamp', 'YES', '', 'CURRENT_TIMESTAMP', 'DEFAULT_GENERATED')


distance and time finding model

In [1]:
import pandas as pd

df = pd.read_excel(r"C:\Users\boopa\Desktop\DS\Drone\delivery.xlsx")
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Type_of_order,Time_taken(min)
0,4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,Snack,24
1,B379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,Snack,33
2,5D6D,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,26
3,7A6A,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,21
4,70A2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,Snack,30


In [4]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius (km)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)

    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

df['distance_km'] = haversine(
    df['Restaurant_latitude'],
    df['Restaurant_longitude'],
    df['Delivery_location_latitude'],
    df['Delivery_location_longitude']
)

In [5]:
df = pd.get_dummies(df, columns=['Type_of_order'], drop_first=True)

In [6]:
X = df.drop(['ID', 'Delivery_person_ID', 'Time_taken(min)'], axis=1)
y = df['Time_taken(min)']

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [8]:
import joblib

joblib.dump(model, "delivery_model.pkl")
joblib.dump(X.columns, "model_columns.pkl")

['model_columns.pkl']

In [9]:
import joblib
cols = joblib.load(r"C:\Users\boopa\Desktop\DS\Drone\model_columns.pkl")
print(list(cols))

['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'distance_km', 'Type_of_order_Drinks ', 'Type_of_order_Meal ', 'Type_of_order_Snack ']


In [13]:
print(df[['Restaurant_latitude','Restaurant_longitude']].drop_duplicates().values.tolist())
print(df[['Delivery_location_latitude','Delivery_location_longitude']].drop_duplicates().values.tolist())

[[22.745049, 75.892471], [12.913041, 77.683237], [12.914264, 77.6784], [11.003669, 76.976494], [12.972793, 80.249982], [17.431668, 78.408321], [23.369746, 85.33982], [12.352058, 76.60665], [17.433809, 78.386744], [30.327968, 78.046106], [10.003064, 76.307589], [18.56245, 73.916619], [30.899584, 75.809346], [26.463504, 80.372929], [19.176269, 72.836721], [12.311072, 76.654878], [18.592718, 73.773572], [17.426228, 78.407495], [22.552672, 88.352885], [18.563934, 73.915367], [23.357804, 85.325146], [12.986047, 80.218114], [19.221315, 72.862381], [13.005801, 80.250744], [26.849596, 75.800512], [21.160522, 72.771477], [12.934179, 77.615797], [18.51421, 73.838429], [11.022477, 76.995667], [21.160437, 72.774209], [15.51315, 73.78346], [15.561295, 73.749478], [0.0, 0.0], [18.55144, 73.804855], [18.593481, 73.785901], [21.173343, 72.792731], [17.451976, 78.385883], [12.972532, 77.608179], [13.064181, 80.236442], [19.121999, 72.908493], [21.149569, 72.772697], [19.091458, 72.827808], [22.539129, 

In [14]:
print(df[['Delivery_person_Age','Delivery_person_Ratings']].mean())

Delivery_person_Age        29.544075
Delivery_person_Ratings     4.632367
dtype: float64
